In [1]:
import requests
import pandas as pd
import os
import json
from dotenv import load_dotenv
import matplotlib.pyplot as plt

In [2]:
load_dotenv()
EPI_API_KEY = os.getenv("EPI_API_KEY")

In [3]:
# API call to database

meta_url = "https://api.eia.gov/v2/electricity/electric-power-operational-data"
params = {"api_key": EPI_API_KEY}

r = requests.get(meta_url, params=params) # 'get' visits the website, returns JSON
print(r.status_code) # printing the status code, 200 = success

meta = r.json() 

print(json.dumps(meta, indent=2)[:4000]) # prints the json so it is readable

200
{
  "response": {
    "id": "electric-power-operational-data",
    "name": "Electric Power Operations (Annual and Monthly)",
    "description": "Monthly and annual electric power operations by state, sector, and energy source.\n    Source: Form EIA-923",
    "frequency": [
      {
        "id": "monthly",
        "description": "One data point for each month.",
        "query": "M",
        "format": "YYYY-MM"
      },
      {
        "id": "quarterly",
        "description": "One data point every 3 months.",
        "query": "Q",
        "format": "YYYY-\"Q\"Q"
      },
      {
        "id": "annual",
        "description": "One data point for each calendar year.",
        "query": "A",
        "format": "YYYY"
      }
    ],
    "facets": [
      {
        "id": "location",
        "description": "State / Census Region"
      },
      {
        "id": "sectorid",
        "description": "Sector"
      },
      {
        "id": "fueltypeid",
        "description": "Energy Source"
   

In [4]:
for f in meta["response"]["frequency"]:
    print(f["id"], f["query"])

monthly M
quarterly Q
annual A


In [5]:
# pull real data
meta_url1 = "https://api.eia.gov/v2/electricity/electric-power-operational-data/data/"

# coal, natural gas, nuclear, hydro, wind, solar.
params1 = {'frequency':'annual',
           'data[]':'generation',
           'facets[location][]':'US',
            'facets[sectorid][]':'99',
            'facets[fueltypeid][]':['COW','NG','NUC','SUN','WND','HYC'],
           'start':'2005',
           'end':'2025',
           'sort[0][column]':'period',
            'sort[0][direction]':'asc',
            'api_key': EPI_API_KEY
           }

r = requests.get(meta_url1, params=params1) # 'get' visits the website, returns JSON
print(r.status_code) # printing the status code, 200 = success

meta = r.json() 



200


In [6]:
print("Total records:", meta["response"]["total"])
print(json.dumps(meta["response"]["data"][:3], indent=2))

Total records: 126
[
  {
    "period": "2005",
    "location": "US",
    "stateDescription": "U.S. Total",
    "sectorid": "99",
    "sectorDescription": "All Sectors",
    "fueltypeid": "COW",
    "fuelTypeDescription": "all coal products",
    "generation": "2012873.04581",
    "generation-units": "thousand megawatthours"
  },
  {
    "period": "2005",
    "location": "US",
    "stateDescription": "U.S. Total",
    "sectorid": "99",
    "sectorDescription": "All Sectors",
    "fueltypeid": "HYC",
    "fuelTypeDescription": "conventional hydroelectric",
    "generation": "270321.25494",
    "generation-units": "thousand megawatthours"
  },
  {
    "period": "2005",
    "location": "US",
    "stateDescription": "U.S. Total",
    "sectorid": "99",
    "sectorDescription": "All Sectors",
    "fueltypeid": "NG",
    "fuelTypeDescription": "natural gas",
    "generation": "760960.25439",
    "generation-units": "thousand megawatthours"
  }
]


In [7]:
eia_df = pd.DataFrame(meta["response"]["data"])
eia_df[eia_df['sectorid'] == '99']

,period,location,stateDescription,sectorid,sectorDescription,fueltypeid,fuelTypeDescription,generation,generation-units
0,2005,US,U.S. Total,99,All Sectors,COW,all coal products,2012873.04581,thousand megawatthours
1,2005,US,U.S. Total,99,All Sectors,HYC,conventional hydroelectric,270321.25494,thousand megawatthours
2,2005,US,U.S. Total,99,All Sectors,NG,natural gas,760960.25439,thousand megawatthours
3,2005,US,U.S. Total,99,All Sectors,NUC,nuclear,781986.365,thousand megawatthours
4,2005,US,U.S. Total,99,All Sectors,SUN,solar,550.29434,thousand megawatthours
...,...,...,...,...,...,...,...,...,...
121,2025,US,U.S. Total,99,All Sectors,HYC,conventional hydroelectric,247023.20955,thousand megawatthours
122,2025,US,U.S. Total,99,All Sectors,NG,natural gas,1807338.28797,thousand megawatthours
123,2025,US,U.S. Total,99,All Sectors,NUC,nuclear,784780.718,thousand megawatthours
124,2025,US,U.S. Total,99,All Sectors,SUN,solar,295671.38248,thousand megawatthours


In [8]:
# seeing the data
print(f'Periods:{eia_df['period'].unique()}')
print(f'Locations: {eia_df['location'].unique()}')
print(f'State Descriptions: {eia_df['stateDescription'].unique()}')
print(f'Sector ID:  {eia_df['sectorid'].unique()}')
print(f'Sector Descriptions:  {eia_df['sectorDescription'].unique()}')
print(f'Fuel Type ID:  {eia_df['fueltypeid'].unique()}')
print(f'Fuel Type Description:  {eia_df['fuelTypeDescription'].unique()}')


Periods:['2005' '2006' '2007' '2008' '2009' '2010' '2011' '2012' '2013' '2014'
 '2015' '2016' '2017' '2018' '2019' '2020' '2021' '2022' '2023' '2024'
 '2025']
Locations: ['US']
State Descriptions: ['U.S. Total']
Sector ID:  ['99']
Sector Descriptions:  ['All Sectors']
Fuel Type ID:  ['COW' 'HYC' 'NG' 'NUC' 'SUN' 'WND']
Fuel Type Description:  ['all coal products' 'conventional hydroelectric' 'natural gas' 'nuclear'
 'solar' 'wind']


In [9]:
print("Total available:", meta["response"]["total"])
print("Rows returned:", len(eia_df))

Total available: 126
Rows returned: 126


In [10]:
eia_df.dtypes

period                 object
location               object
stateDescription       object
sectorid               object
sectorDescription      object
fueltypeid             object
fuelTypeDescription    object
generation             object
generation-units       object
dtype: object

In [14]:
# converting data
eia_df['period'] = eia_df['period'].astype(int)
eia_df['generation'] = eia_df['generation'].astype(float)

eia_df.head(20)


,period,location,stateDescription,sectorid,sectorDescription,fueltypeid,fuelTypeDescription,generation,generation-units
0,2005,US,U.S. Total,99,All Sectors,COW,all coal products,2.012873e+06,thousand megawatthours
1,2005,US,U.S. Total,99,All Sectors,HYC,conventional hydroelectric,2.703213e+05,thousand megawatthours
2,2005,US,U.S. Total,99,All Sectors,NG,natural gas,7.609603e+05,thousand megawatthours
3,2005,US,U.S. Total,99,All Sectors,NUC,nuclear,7.819864e+05,thousand megawatthours
4,2005,US,U.S. Total,99,All Sectors,SUN,solar,5.502943e+02,thousand megawatthours
5,2005,US,U.S. Total,99,All Sectors,WND,wind,1.781055e+04,thousand megawatthours
6,2006,US,U.S. Total,99,All Sectors,COW,all coal products,1.990511e+06,thousand megawatthours
7,2006,US,U.S. Total,99,All Sectors,HYC,conventional hydroelectric,2.892464e+05,thousand megawatthours
8,2006,US,U.S. Total,99,All Sectors,NG,natural gas,8.164408e+05,thousand megawatthours
9,2006,US,U.S. Total,99,All Sectors,NUC,nuclear,7.872186e+05,thousand megawatthours


In [15]:
# carbon intensity
eia_carbon_intensity_df = eia_df.copy()
eia_carbon_intensity_df.groupby('period')['generation'].sum()
eia_carbon_intensity_df.head(20)

,period,location,stateDescription,sectorid,sectorDescription,fueltypeid,fuelTypeDescription,generation,generation-units
0,2005,US,U.S. Total,99,All Sectors,COW,all coal products,2.012873e+06,thousand megawatthours
1,2005,US,U.S. Total,99,All Sectors,HYC,conventional hydroelectric,2.703213e+05,thousand megawatthours
2,2005,US,U.S. Total,99,All Sectors,NG,natural gas,7.609603e+05,thousand megawatthours
3,2005,US,U.S. Total,99,All Sectors,NUC,nuclear,7.819864e+05,thousand megawatthours
4,2005,US,U.S. Total,99,All Sectors,SUN,solar,5.502943e+02,thousand megawatthours
5,2005,US,U.S. Total,99,All Sectors,WND,wind,1.781055e+04,thousand megawatthours
6,2006,US,U.S. Total,99,All Sectors,COW,all coal products,1.990511e+06,thousand megawatthours
7,2006,US,U.S. Total,99,All Sectors,HYC,conventional hydroelectric,2.892464e+05,thousand megawatthours
8,2006,US,U.S. Total,99,All Sectors,NG,natural gas,8.164408e+05,thousand megawatthours
9,2006,US,U.S. Total,99,All Sectors,NUC,nuclear,7.872186e+05,thousand megawatthours
